# Task 1 Statistics Evidence - xuyu8020

This notebook records the Task 1 contribution for `xuyu8020`. It demonstrates how the cleaned NSW region summary dataset is used to calculate five individual derived statistics.

The five statistics focus on population structure, gender balance, income growth, income distribution, and business activity in New South Wales.

## Setup

This section prepares the notebook environment. The project root is set as the working directory so that all relative paths work correctly, even when this notebook is opened from the `notebooks/xuyu8020` folder.

The notebook then loads the project settings from `configs/local.yaml` and imports the statistics module for `xuyu8020`. This allows the notebook to directly call the five statistics functions implemented in `xuyu8020_statistics.py`.

Before running this notebook, make sure:

- `uv sync` has already been run.
- The project virtual environment is selected as the notebook kernel.
- `configs/local.yaml` exists.
- The Task 1 raw CSV file is available in the expected project data folder.

In [ ]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "xuyu8020"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

## Shared Cleaning Input

This section runs the shared Task 1 cleaning workflow. The original NSW region summary CSV contains mixed column names, year columns, descriptions, units, missing values, and values stored as text. Therefore, the data must be cleaned before the derived statistics are calculated.


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

## Individual Derived Statistics

This section calculates the five derived statistics implemented in `xuyu8020_statistics.py`.

Each statistic function receives the cleaned Task 1 dataframe and returns a `StatisticResult` containing:

- `statistic_id`
- `title`
- `value`
- `unit`
- `description`

The purpose of these statistics is not just to repeat raw values from the CSV, but to create more interpretable indicators using ratios, growth rates, gaps, or net rates.


In [ ]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({
            "function": statistic_function.__name__,
            "error": f"{type(exc).__name__}: {exc}",
        })
        continue

    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 200)

display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
    .style
    .set_properties(**{
        "text-align": "left",
        "white-space": "normal",
        "vertical-align": "top",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "left"),
                ("vertical-align", "top"),
            ],
        },
    ])
)

if errors:
    display(pd.DataFrame(errors))

## Explanation Notes

This section explains the five derived statistics for `xuyu8020`. They are based on the shared cleaned NSW region summary dataset.

- `xuyu8020-1`: The age dependency ratio in 2024 is 54.44 dependents per 100 working-age persons. This means that for every 100 people aged 15-64, there are about 54 people outside the main working-age group. This is useful because a higher dependency ratio may indicate greater demand for services such as education, healthcare, and aged care.

- `xuyu8020-2`: The sex ratio in 2024 is 98.98 males per 100 females. This shows that the female population is slightly larger than the male population in NSW.

- `xuyu8020-3`: Median total income increased by 15.16% from 2018 to 2022. This gives a clearer interpretation than simply reporting the raw median income values, because it shows income change over time.

- `xuyu8020-4`: Mean total income is 37.03% higher than median total income in 2022. Since the mean is higher than the median, this suggests that the income distribution is right-skewed, with higher-income earners pulling the average upward.

- `xuyu8020-5`: The business net entry rate in 2024 is 2.95% of total businesses. This means that business entries exceeded business exits, suggesting positive business formation relative to the existing business base.

Overall, these five statistics provide demographic and economic context for the later POI-based resource analysis. They show that resource demand and resource distribution should not be interpreted only through geography, but also through population structure, income patterns, and local economic activity.